### IntelliCast - Experiments

In [ ]:
import os
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from typing import List
from langchain.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
import torch
from dotenv import load_dotenv
load_dotenv()
from pinecone import Pinecone
from pinecone import ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_openai import ChatOpenAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Extract text from pdf file
def load_pdf(data_path):
    loader = DirectoryLoader(
        data_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader #type: ignore
    )
    
    documents = loader.load()
    return documents

In [ ]:
extracted_data = load_pdf("C:/Users/Aadarsh/Desktop/IntelliCast/data")

In [ ]:
extracted_data

In [ ]:
len(extracted_data)

In [ ]:
# Filter the unwanted metadata of extracted_data
def filter_to_minimal_docs(docs: List[Document])-> List[Document]:
    """
    Given a liat of Documents objects, return a new list of Document objects
    obtaining only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [ ]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [ ]:
minimal_docs

In [ ]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [ ]:
texts_chunks = text_split(minimal_docs)

In [ ]:
print(f"Number of chunks: {len(texts_chunks)}")

In [ ]:
texts_chunks

In [ ]:
# Embedding
def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"} #type: ignore
    )
    return embeddings

In [ ]:
embedding = download_embeddings()

In [ ]:
embedding

In [ ]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY #type: ignore
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY #type: ignore

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
pc

In [ ]:
index_name = "intellicast"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [ ]:
docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunks,
    embedding=embedding,
    index_name=index_name
)

In [ ]:
# Load existing index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

In [ ]:
# Add more data to the existing pinecone index
dummy_doc = Document(
    page_content="I love you.",
    metadata={"source": "Youtube"}
)

In [ ]:
docsearch.add_documents(documents=[dummy_doc])

In [ ]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [ ]:
retrieved_docs = retriever.invoke("What is Acne?")

In [ ]:
retrieved_docs

In [ ]:
gpt_model = ChatOpenAI(model="gpt-5.4-mini-2026-03-17")

In [ ]:
system_prompt = (
    "You are a medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise. "
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),    
    ]
)

In [ ]:
question_answer_chain = create_stuff_documents_chain(gpt_model, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [ ]:
response = rag_chain.invoke({"input": "What is acromegaly and giganstims?"})
print(response["answer"])

In [ ]:
response = rag_chain.invoke({"input": "What is cancer?"})
print(response["answer"])